In [1]:

    
import numpy as np
nsites = 8
nv = 2
sh = 6
trainfeatures = np.load("trainfeatures.npy")[sh::100, :nv, :].reshape(850, nsites).T
testfeatures = np.load("testfeatures.npy")[sh::100, :nv, :].reshape(120, nsites).T
traintargets = np.load("traintargets.npy")[sh::100, :nv, :].reshape(850, nsites).T
testtargets = np.load("testtargets.npy")[sh::100, :nv, :].reshape(120, nsites).T

#"pauli_prop_values.h5", "w") do file
#    write(file, "qrc_train", expectations)
#    write(file, "qrc_test", expectations_test)
#    write(file, "test_targets", testtargets)
#    write(file, "train_targets", traintargets)
#    write(file, "test_features", testfeatures)
#    write(file, "train_features", trainfeatures)

In [2]:
import numpy as np
statsfeatures = np.zeros((970, nsites))
allpredict = np.zeros((970, nsites))
statsfeatures[:850,:] = trainfeatures.T
statsfeatures[850:,:] = testfeatures.T
allpredict[:850,:] = traintargets.T
allpredict[850:,:] = testtargets.T

In [3]:
train_embeddings = np.load("train_embeddings_noiseless.npy")
test_embeddings = np.load("test_embeddings_noiseless.npy")
#train_embeddings = np.load("train_embeddings_noisy.npy")
#test_embeddings = np.load("test_embeddings_noisy.npy")
#train_embeddings = np.load("train_embeddings_noisy2.npy")
#test_embeddings = np.load("test_embeddings_noisy2.npy")
#train_embeddings = np.load("train_embeddings_noisy4.npy")
#test_embeddings = np.load("test_embeddings_noisy4.npy")
#train_embeddings = np.load("train_embeddings_noiselessl.npy")
#test_embeddings = np.load("test_embeddings_noiselessl.npy")
train_embeddings.shape

(850, 780)

In [4]:
train_embeddings = np.load("train_embeddings_noiseless.npy")
test_embeddings = np.load("test_embeddings_noiseless.npy")
#test_embeddingsn = np.load("test_embeddings_noiselessl.npy")
#print(test_embeddingsn[:,2])
#print(test_embeddings[:,2])
quantum_train = np.zeros((850,360))
quantum_test = np.zeros((120,360))

quantum_train = train_embeddings
quantum_test = test_embeddings[:,:]

In [5]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

all_r2 = np.zeros(8)
all_mse = np.zeros(8)
all_r2_qrc = np.zeros(8)
all_mse_qrc = np.zeros(8)
imps = []
ma = []
mb = []
n_windows = 1
bs = 3
ws = test_embeddings.shape[1]//10
for l in range(nsites):
    # Train simle linear
    X_Train = statsfeatures[:850]
    X_Test = statsfeatures[850:]
    Y_Train = allpredict[:850,l] #np.concatenate((traintargets[:,i,:], traintargets[:,j,:], traintargets[:,k,:]), axis=1)[:,:]
    Y_Test = allpredict[850:,l] #np.concatenate((testtargets[:,i,:], testtargets[:,j,:], testtargets[:,k,:]), axis=1)[:,:]
    alphas = np.logspace(-3, 4, 40)   # 100 candidates from 0.0001 to 10000
    from sklearn.linear_model import Ridge, RidgeCV
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=10,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
)
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train)
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2[l] = r2
    all_mse[l] = mse 
    alpha_best1 = alpha_best
    #print(X_Train.shape)
    
    # Train QRC
    X_Train = np.concatenate((X_Train[:], quantum_train[:850,ws*bs+12:ws*bs+ws*n_windows-0]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #print(X_Train.shape)
    X_Test = np.concatenate((X_Test, quantum_test[:,ws*bs+12:ws*bs+ws*n_windows-0]), axis=1)#, quantum_test[:,-ws*n_windows:]), axis=1)
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=10,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
    )
    
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    #print(alpha_best1, alpha_best)
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train[:])
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2_qrc[l] = r2
    all_mse_qrc[l] = mse
    if all_r2[l] > -0.5 and ((l < 12) or (l > 35 and l < (36+12))):        
        print(f"{l} {all_r2[l]}          {all_r2_qrc[l]}       {all_r2[l]-all_r2_qrc[l]}")
        imps += [all_r2[l]-all_r2_qrc[l]]
        ma += [all_r2[l]]
        mb += [all_r2_qrc[l]]
    
# print(np.mean(imps), np.std(imps), np.mean(ma), np.mean(mb))

0 0.7945400109322451          0.8233926580715711       -0.028852647139326004
1 0.44341715485000066          0.3706132399791856       0.07280391487081506
2 0.33286528758274914          0.22094226575970455       0.11192302182304459
3 -0.050701632245132355          -0.37968107407082985       0.3289794418256975
4 -0.11972723006503472          0.038879107058103934       -0.15860633712313865
5 0.8374172213209484          0.8337673988911809       0.003649822429767524
6 0.022753032159241626          -0.04807201559883678       0.07082504775807841
7 0.12482595415183151          0.42629871370913264       -0.30147275955730113


In [7]:
from pathlib import Path
import re
import json
def load_test_files(directory: str = ".") -> list[tuple[int, Path, dict]]:
    """Find and load all test JSON files, sorted by test number."""
    dir_path = Path(directory)
    pattern = re.compile(r"^test_(\d+)__[a-zA-Z0-9]+\.json$")
 
    matches = []
    for file in dir_path.iterdir():
        m = pattern.match(file.name)
        if m:
            matches.append((int(m.group(1)), file))
 
    matches.sort(key=lambda x: x[0])
 
    #results = []
    #for number, path in matches:
    #    with path.open("r", encoding="utf-8") as f:
    #        data = json.load(f)
    #    results.append((number, path, data))
 
    return matches

def load_train_files(directory: str = ".") -> list[tuple[int, Path, dict]]:
    """Find and load all test JSON files, sorted by test number."""
    dir_path = Path(directory)
    pattern = re.compile(r"^train_(\d+)__[a-zA-Z0-9]+\.json$")
 
    matches = []
    for file in dir_path.iterdir():
        m = pattern.match(file.name)
        if m:
            matches.append((int(m.group(1)), file))
 
    matches.sort(key=lambda x: x[0])
 
    #results = []
    #for number, path in matches:
    #    with path.open("r", encoding="utf-8") as f:
    #        data = json.load(f)
    #    results.append((number, path, data))
 
    return matches

test_files = load_test_files("aquila/results")
train_files = load_train_files("aquila/results")

In [8]:
t1 = 70
t2 = 69
t1 = 0
t2 = 0
aquila_features = np.zeros((850-t1, 36))
aquila_test_features = np.zeros((120-t2, 36))
import json

pos = -1

for m in range(t1, 850):
    t = 0
    with train_files[m][1].open("r", encoding="utf-8") as f:
        data = json.load(f)
        #print(data['measurements'][1]['shotResult'])
    pos += 1
    for d in range(8):
        s = 0
        t = 0
        value = 0
        for i in range(15):
            key = data['measurements'][i]['shotResult']['postSequence']
            pre = data['measurements'][i]['shotResult']['preSequence']
            for k in range(6):
                if sum(pre[k*8:k*8+8]) != 8:
                    t += 1
                    
                    if key[k*8+d] == 0:
                        s -= 1
                    else:
                        s += 1
        aquila_features[m-t1,d] = s/t
    p = 7
    for q1 in range(8):
        for q2 in range(q1+1,8):
            p += 1
            s = 0
            t = 0
            for i in range(15):
                key = data['measurements'][i]['shotResult']['postSequence']
                pre = data['measurements'][i]['shotResult']['preSequence']
                for k in range(6):
                    if sum(pre[k*8:k*8+8]) != 8:
                        t += 1
                        if key[k*8+q1] == key[k*8+q2]:
                            s += 1
                        else:
                            s -= 1
            aquila_features[m-t1,p] = s/t


for m in range(t2,120):
    t = 0
    with test_files[m][1].open("r", encoding="utf-8") as f:
        data = json.load(f)
        #print(data['measurements'][1]['shotResult'])
    pos = pos + 1
    for d in range(8):
        s = 0
        t = 0
        value = 0
        for i in range(15):
            key = data['measurements'][i]['shotResult']['postSequence']
            pre = data['measurements'][i]['shotResult']['preSequence']
            for k in range(6):
                if sum(pre[k*8:k*8+8]) != 8:
                    t += 1
                    
                    if key[k*8+d] == 0:
                        s -= 1
                    else:
                        s += 1
        #print(d, s/(t), t)
        aquila_test_features[m-t2,d] = s/t
    
    p = 7
    for q1 in range(8):
        for q2 in range(q1+1,8):
            p += 1
            for i in range(15):
                key = data['measurements'][i]['shotResult']['postSequence']
                pre = data['measurements'][i]['shotResult']['preSequence']
                for k in range(6):
                    if sum(pre[k*8:k*8+8]) != 8:
                        t += 1
                        if key[k*8+q1] == key[k*8+q2]:
                            s += 1
                        else:
                            s -= 1
            aquila_test_features[m-t2,d] = s/t


In [9]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

labels = ["D mean", "D std", "D skew", "D kurt", "D_p mean", "D_p std", "D_p skew", "D_p kurt"]
all_r2 = np.zeros(12)
all_mse = np.zeros(12)
all_r2_qrc = np.zeros(12)
all_mse_qrc = np.zeros(12)
imps = []
ma = []
mb = []
n_windows = 1
bs = 3
ws = test_embeddings.shape[1]//10
for l in range(8):
    # Train simle linear
    X_Train = statsfeatures[t1:850,:]#statsfeatures[:70]
    X_Test = statsfeatures[850+t2:]
    Y_Train = allpredict[t1:850,l] #np.concatenate((traintargets[:,i,:], traintargets[:,j,:], traintargets[:,k,:]), axis=1)[:,:]
    Y_Test = allpredict[850+t2:,l] #np.concatenate((testtargets[:,i,:], testtargets[:,j,:], testtargets[:,k,:]), axis=1)[:,:]
    alphas = np.logspace(-5, 4, 100)   # 100 candidates from 0.0001 to 10000
    from sklearn.linear_model import Ridge, RidgeCV
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=14,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
)
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train)
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2[l] = r2
    all_mse[l] = mse 
    alpha_best1 = alpha_best
    #print(X_Train.shape)
    
    # Train QRC
    X_Train = np.concatenate((X_Train[:], aquila_features[:,8:]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #print(X_Train.shape)
    X_Test = np.concatenate((X_Test, aquila_test_features[:,8:]), axis=1)#, quantum_test[:,-ws*n_windows:]), axis=1)
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=14,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
    )
    
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    # print(alpha_best1, alpha_best)
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-7, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train[:])
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2_qrc[l] = r2
    all_mse_qrc[l] = mse 
    if all_r2[l] > -0.5 and ((l < 12) or (l > 35 and l < (36+12))):
        
        print(f"{labels[l]} & {all_r2[l]:4.3}   &       {all_r2_qrc[l]:4.3}  &     {all_r2[l]-all_r2_qrc[l]:4.3} \\\\")
        imps += [all_r2[l]-all_r2_qrc[l]]
        ma += [all_r2[l]]
        mb += [all_r2_qrc[l]]
    
#print(np.mean(imps), np.std(imps), np.mean(ma), np.mean(mb))

D mean & 0.795   &       0.796  &     -0.0016 \\
D std & 0.443   &       0.446  &     -0.00244 \\
D skew & 0.333   &       0.326  &     0.00658 \\
D kurt & -0.0506   &       -0.0511  &     0.000519 \\
D_p mean & -0.141   &       0.029  &     -0.17 \\
D_p std & 0.837   &       0.834  &     0.00322 \\
D_p skew & 0.0224   &       0.0226  &     -0.000235 \\
D_p kurt & 0.125   &       0.119  &     0.00596 \\


In [10]:
np.save('aquila_features', aquila_features)
np.save('aquila_test_features', aquila_test_features)